# 🧪 Debug Notebook: Espejo del Modelo de Adopción (AdoptionModel2_0)
Este notebook permite probar el modelo de adopción replicado manualmente, sin depender de su importación desde el proyecto principal. Puedes insertar cualquier entrada manualmente y obtener todas las salidas paso a paso.

In [6]:
import pandas as pd
import numpy as np
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:

class AdoptionModel2_0_Debug:
    def __init__(self, state, growth, demand_area, technologies_df, fuel_price_plan=[], app_price_plan=[], deploy_caps={}):
        self.state = state
        self.growth = growth
        self.demand_area = demand_area
        self.technologies = technologies_df.set_index('Technologies_id')
        self.price_plans_fuels = fuel_price_plan
        self.price_plans_appl = app_price_plan
        self.deployment_plan_appliances_max_cap = deploy_caps.get("appliances", [])
        self.deployment_plan_fuels_max_cap = deploy_caps.get("fuels", [])
        self.potential_adoption = {}

    def simulate_state(self):
        """ Simulate the state of the demand area. Adjusts the initial adoptions based on the previous state. """
        try: 
            self.demand_area.data.setdefault('initial_adoptions', {'rural': {}, 'urban': {}})
            base_year = self.growth.base_year
            year = self.state.year
            semester = self.state.semester

            if (year > base_year) or (year == base_year and semester == "second"):
                self.updateInitAdoption(self.prev_state)

            self.save_results_init_to_state()

            if semester == "second" or self.prev_state is None:
                self.current_price_plan = self.state.price_plan.details
                self.current_deployment_plan = self.state.deployment_plan.details
            else:
                self.current_price_plan = self.prev_state.price_plan.details
                self.current_deployment_plan = self.prev_state.deployment_plan.details
        except Exception as e:
            logging.error("Error en la simulación del estado: %s", str(e), exc_info=True)
            raise

    def updateInitAdoption(self, prev_state):
        try:
            tech_id_to_name = self.all_technologies["Tech_name"].to_dict()  # <- usar todas
            for area_type in self.area_types:
                prev_potential = prev_state.get_potential_adoption(self.demand_area.id, area_type)
                if not isinstance(prev_potential, dict):
                    continue
                self.demand_area.data['initial_adoptions'].setdefault(area_type, {})
                for tech_id, value in prev_potential.items():
                    tech_name = tech_id_to_name.get(tech_id)
                    if tech_name:
                        self.demand_area.data['initial_adoptions'][area_type][tech_name] = value
        except Exception as e:
            logging.error("Error actualizando adopciones iniciales: %s", e, exc_info=True)
            raise

    def save_results_init_to_state(self):
        try:
            for area_type in self.area_types:
                adoptions = self.demand_area.data['initial_adoptions'].get(area_type, {})
                full_tech_ids = self.all_technologies.index.tolist()
                full_tech_names = self.all_technologies["Tech_name"]

                values_by_id = {
                    tid: adoptions.get(full_tech_names[tid], 0.0)
                    for tid in full_tech_ids
                }

                self.state.set_initial_adoption(self.demand_area.id, area_type, values_by_id)
        except Exception as e:
            logging.error("Error guardando adopciones iniciales: %s", e, exc_info=True)
            raise

    def _update_social_params(self, area):
        year_diff = self.state['year'] - self.growth['base_year']
        u = 1 if area == 'urban' else 0
        #params = self.demand_area['aggregated_clusters'][area]['params'].copy()
        params = self.demand_area.data['aggregated_clusters'][area]['params'].copy()

        df_mult = self.growth['dem_mult'][u]
        df_elast = self.growth['dem_elast'][u]
        df_sc = self.growth['soc_clus'][u]
        params['DemandMult'] = {
            'Electricity': params.get('DemandMult', {}).get('Electricity', 1.0) * (1 + df_mult['Electricity']) ** year_diff,
            'Cooking':    params.get('DemandMult', {}).get('Cooking',    1.0) * (1 + df_mult['Cooking']) ** year_diff,
            'Heating':    params.get('DemandMult', {}).get('Heating',    1.0) * (1 + df_mult['Heating']) ** year_diff,
        }
        params['e_elast_demand'] *= (1 + df_elast['Electricity']) ** year_diff
        params['Population'] *= (1 + df_sc['Population_growth']) ** year_diff
        params['will_pay'] *= (1 + df_sc['Income_growth']) ** year_diff
        params['invest_cap'] *= (1 + df_sc['Income_growth']) ** year_diff
        cf = (1 + df_sc['Technology_progress']) ** year_diff
        params['change_fact'] = min(1.0, params['change_fact'] * cf)
        params['better_fact'] *= cf
        params['worse_fact'] *= cf
        params['social_weight'] = min(1.0, params['social_weight'] * (1 + df_sc['Social_weight']) ** year_diff)
        sb = params.get('social_balance', {})
        new_sb = {}
        total = 0.0
        for key, growth_key in [('health','Health'),('time_gender','Time_gen'),('emissions','Emissions')]:
            val = sb.get(key, 0.0) * (1 + df_sc[growth_key]) ** year_diff
            new_sb[key] = val
            total += val
        new_sb['deforestation'] = max(1.0 - total, 0.0)
        if sum(new_sb.values()) > 0:
            new_sb = {k: v/sum(new_sb.values()) for k,v in new_sb.items()}
        params['social_balance'] = new_sb
        return params

    def _compute_indicators(self, area, soc_params):
        """Calculate indicators for the given area and compute projection weights."""
        try:
            tech = self.technologies.copy()
            init = pd.Series(self.demand_area.data['initial_adoptions'][area])
            init = init.reindex(tech['Tech_name']).fillna(0).values
            logging.info("Adopciones iniciales: %s", init)
            ref_health = (init * tech['Health']).sum()
           
            ref_time = (init * (tech['FuelTimeGen'] + tech['ApplianceTimeGen'])).sum()
            
            ref_emis = (init * tech['Emissions']).sum()
            
            ref_def = (init * tech['Deforestation']).sum()
            
            ref = np.maximum([ref_health, ref_time, ref_emis, ref_def], 0.0)
            print("Ref: %s", ref)
            # Reindex timegender to match tech_names
            timegen_series = self.demand_area.data['time_gen_modified'].get(area, {})
            timegenmodified = pd.Series(timegen_series).reindex(tech['Tech_name'])#.fillna(1.0).values
            #print("TimeGenModified: %s", timegenmodified)


            ref_health = (init * tech['Health']).sum()
            ref_time   = (init * timegenmodified).sum()#(init * timegenmodified).sum()
                                                    
            ref_emis   = (init * tech['Emissions']).sum()
            ref_def    = (init * tech['Deforestation']).sum()

            ref = np.maximum([ref_health, ref_time, ref_emis, ref_def], 0.0)

            weights = soc_params['social_balance']
            S = (
                tech['Health'] / ref[0] * weights['health'] +
                (tech['FuelTimeGen'] + tech['ApplianceTimeGen']) / ref[1] * weights['time_gender'] +
                tech['Emissions'] / ref[2] * weights['emissions'] +
                tech['Deforestation'] / ref[3] * weights['deforestation']
            )

            biomass_multipliers = self.demand_area.data.get("biomass_multipliers", {}).get("loc_price", {})
            fuel_names = tech['Fuel_id'].map(self.fuel_id_to_name)
            appl_names = tech['Appliance_id'].map(self.appl_id_to_name)

            fuel_loc_multiplier = fuel_names.map(lambda name: biomass_multipliers.get(name, {}).get(area, 1.0))
            fuel_plan_multiplier = fuel_names.map(lambda name: next((p.get(name, 1.0) for p in self.price_plans_fuels), 1.0))
            appl_plan_multiplier = appl_names.map(lambda name: next((p.get(name, 1.0) for p in self.price_plans_appl), 1.0))

            p_fuel = tech['FuelPrice'] * fuel_loc_multiplier * fuel_plan_multiplier
            p_appl = tech['AppliancePrice'] * appl_plan_multiplier
            time_gen = self.demand_area.data['time_gen_modified'].get(area, pd.Series(1.0, index=tech['Tech_name']))
            time_vec = tech['Tech_name'].map(time_gen).fillna(1.0).values
            p_time = soc_params.get('invest_cap', 1.0) * soc_params.get('will_pay', 1.0) * time_vec
            C = p_fuel + p_appl + p_time  # Este es el relavive prices 

            # PROJECTION WEIGHTS explícitos
            soc_weight = soc_params.get("social_weight", 1.0)
            better_fact = soc_params.get("better_fact", 1.0)
            worse_fact = soc_params.get("worse_fact", 1.0)
            #C es absolute pries pero necesito relative prices 
            C_rel_factor = (C * init).sum()  # Evitar división por cero

            C_rel_factor = max((C_rel_factor + soc_params.get('will_pay', 0))/2 , 0.01)
            C_rel = C/ max(C_rel_factor, 0.0001) #para cada tecnología su precio absoluto (C) dividido por máximo entre el factor de referencia y 0.01
            

            soc_econ_rep = soc_weight * S + (1 - soc_weight) * C_rel#C ·Proyeccion socio económica 
            max_soc_econ_rep = np.clip(soc_econ_rep, 0.01, None)

            projection_weights = np.where(
                soc_econ_rep <= 1,
                better_fact * (1 / max_soc_econ_rep - 1),
                1/(1 + worse_fact * (soc_econ_rep - 1))
            )

            df = pd.DataFrame({
                'C': C,
                'S': S,
                'W': projection_weights
            }, index=tech.index)

            self.projection_weights = getattr(self, "projection_weights", {})
            self.projection_weights[area] = pd.Series(projection_weights, index=tech.index)

            return df
        except Exception as e:
            logging.error("Error calculando indicadores: %s", e, exc_info=True)
            raise

    def _compute_init_candidates(self, df, area):
        """
        Compute initial adoption candidates with appliance price penalty and projection weights.
        """
        try:
            tech = self.technologies
            init0 = pd.Series(self.demand_area.data['initial_adoptions'][area])
            init0 = init0.reindex(tech['Tech_name']).fillna(0).values

            #Lo que falta que llamaríamos penalties y normalizaciones 
            #Final appl purchase prices = appliance price multiplier * appliance price plan
            # Reduction factor aappliance price = Si (Final appl purchase prices < 0.01) entonces 0.01  sino 
            # si ( Final appl purchase prices <  investment capacity) entonces min(1; 0.5 * sqrt(investment capacity / Final appl purchase prices) y 
            # sino 0.5 * (investment cpaacity/Final appl purchase prices)^2          
            # Reduced Vaalue applaance price = Reduction factor aappliance price * projection weights
            # Reduced value appliance price  normalized = Reduced Vaalue applaance price / sum(reduced value appliance price)

            

            change = self.demand_area.data['aggregated_clusters'][area]['params']['change_fact']
            tau = max(1, (2 * (self.state['year'] - self.growth['base_year'])) ** 2)

            # AppliancePrice penalty
            appl_price = tech['AppliancePrice'].clip(lower=0.01)
            invest_cap = self.demand_area.data['aggregated_clusters'][area]['params'].get('invest_cap', 1.0)
            final_appl_purchase_prices = appl_price * tech['AppliancePrice'].map(self.appl_id_to_name).map(lambda name: next((p.get(name, 1.0) for p in self.price_plans_appl), 1.0))
            red_factor_appl_price = np.where(final_appl_purchase_prices < 0.01, 0.01,
                np.where(final_appl_purchase_prices < invest_cap,
                         np.minimum(0.5 * np.sqrt(invest_cap / final_appl_purchase_prices), 1.0),
                         0.5 * (invest_cap / final_appl_purchase_prices) ** 2))
            reduced_value_appl_price = red_factor_appl_price * df['W']
            reduced_value_appl_price_normalized = reduced_value_appl_price / reduced_value_appl_price.sum()


            #reduction_factor = np.minimum(np.sqrt(invest_cap / appl_price) * 0.5, 1.0)# stáa aquí el errpr 

            

            # Use projection weights W (already computed)
            # r_appl = df['W'] * reduction_factor
            # # Normalize r_appl
            # r_appl_sum = r_appl.sum()
            # if r_appl_sum > 1e-9:
            #     r_appl = r_appl / r_appl_sum

            # Reduced factor for new technology = si (iniit0 < 0.05) entonces change + 20 * (1 - change * init0) sino 1.0
            #Reduced value for neew technology = init0 + (reduced value appliance price normaalized - init0) * reduced factor for new technology
            # Redued value for new technology normalized = reduced value for neew technology / sum(reduced value for new technology)
            # Initial candidates = (10 * init0 + reduced value for new technology normalized * tau) / (10 + tau)

            f_new = np.where(init0 < 0.05, change + 20 * (1 - change * init0), 1.0)
            n = init0 + (reduced_value_appl_price_normalized - init0) * f_new
            n_normalized = n / n.sum() if n.sum() > 1e-9 else n

            init_cand = (10 * init0 + n_normalized * tau) / (10 + tau)
            return init_cand
        except Exception as e:
            logging.error("Error calculando candidatos iniciales: %s", e, exc_info=True)
            raise
    
    def _adoption_lp(self, init_cand, area):
        """
        Alternativa iterativa inspirada en el modelo heurístico anterior.
        Distribuye adopción evitando concentrarla en una sola tecnología.
        """
        try: 
            tech_ids = self.technologies.index
            appl_ids = self.technologies['Appliance_id']
            fuel_ids = self.technologies['Fuel_id']
            is_urban = (area == "urban")

            appl_cap_dict = next((item for item in self.deployment_plan_appliances_max_cap if item.get("Is_Urban") == is_urban), {})
            fuel_cap_dict = next((item for item in self.deployment_plan_fuels_max_cap if item.get("Is_Urban") == is_urban), {})

            initial = dict(zip(tech_ids, init_cand))
            reduced_appl = {}
            reduced_fuel = {}
            candidate = {}
            excess = {}
            last_candidate = {tid: val for tid, val in initial.items()}

            tech_list = list(zip(tech_ids, fuel_ids, appl_ids))

            for tid, _, aid in tech_list:
                cap = appl_cap_dict.get(aid, 1.0)
                reduced_appl[tid] = min(initial[tid], cap)
            sum_appl = sum(reduced_appl.values())
            if sum_appl > 1:
                for tid in reduced_appl:
                    reduced_appl[tid] /= sum_appl

            for _ in range(50):
                fuel_sums = {}
                for tid, fid, _ in tech_list:
                    fuel_sums[fid] = fuel_sums.get(fid, 0.0) + reduced_appl.get(tid, 0.0)

                reduced_fuel = {}
                for tid, fid, _ in tech_list:
                    val = reduced_appl.get(tid, 0.0)
                    cap = fuel_cap_dict.get(fid, 1.0)
                    group_sum = fuel_sums.get(fid, 1.0)
                    r = min(val, val * cap / group_sum if group_sum > 0 else 0.0)
                    reduced_fuel[tid] = max(0.0, min(1.0, r))

                sum_rf = sum(reduced_fuel.values())
                if sum_rf > 1:
                    for tid in reduced_fuel:
                        reduced_fuel[tid] /= sum_rf

                total_excess = 0.0
                total_candidates = 0.0
                for tid in initial:
                    val_init = initial[tid]
                    val_rf = reduced_fuel.get(tid, 0.0)
                    excess[tid] = max(0.0, val_init - min(val_init, val_rf))
                    if val_rf < val_init or last_candidate[tid] < 1e-12:
                        candidate[tid] = 0.0
                    else:
                        candidate[tid] = val_rf
                    total_excess += excess[tid]
                    total_candidates += candidate[tid]

                if total_excess < 1e-9 or total_candidates < 1e-9:
                    break

                for tid in initial: #Revisar esta parte me  suena muy chino 
                    val_rf = reduced_fuel.get(tid, 0.0)
                    delta = (candidate[tid] * total_excess / total_candidates) if total_candidates > 0 else 0.0 #candidate[tid] * excess[tid] / total_excess if total_excess > 0 else 0.0
                    initial[tid] = max(0.0, min(1.0, val_rf + delta))
                    last_candidate[tid] = candidate[tid]

                for tid, _, aid in tech_list:
                    cap = appl_cap_dict.get(aid, 1.0)
                    reduced_appl[tid] = min(initial[tid], cap)
                sum_appl = sum(reduced_appl.values())
                if sum_appl > 1:
                    for tid in reduced_appl:
                        reduced_appl[tid] /= sum_appl

            sum_final = sum(reduced_fuel.values())
            if sum_final > 1:
                for tid in reduced_fuel:
                    reduced_fuel[tid] /= sum_final

            return pd.Series(reduced_fuel)
        except Exception as e:
            logging.error("Error en la adopción LP: %s", e, exc_info=True)
            raise



In [8]:
import pandas as pd
import numpy as np

# --- Tecnologías (simplificadas a 3 para ejemplo compacto) --- 0.017	0.017	0.017	0.050	0.050	1.000	0.833	0.667	1.000	0.833	0.667	1.000	0.833	0.667	0.333	1.000	0.833	0.667	0.050
import pandas as pd

technologies_df = pd.DataFrame([
    {"Technologies_id": 1, "Tech_name": "El0", "FuelPrice": 0.199, "AppliancePrice": 0.004, "Health": 0.000, "FuelTimeGen": 0.000, "ApplianceTimeGen": 0.017, "Emissions": 0.222, "Deforestation": 0.000},
    {"Technologies_id": 2, "Tech_name": "El1", "FuelPrice": 0.142, "AppliancePrice": 0.008, "Health": 0.000, "FuelTimeGen": 0.000, "ApplianceTimeGen": 0.017, "Emissions": 0.159, "Deforestation": 0.000},
    {"Technologies_id": 3, "Tech_name": "El2", "FuelPrice": 0.105, "AppliancePrice": 0.012, "Health": 0.000, "FuelTimeGen": 0.000, "ApplianceTimeGen": 0.017, "Emissions": 0.117, "Deforestation": 0.000},
    {"Technologies_id": 4, "Tech_name": "LPG", "FuelPrice": 0.074, "AppliancePrice": 0.012, "Health": 0.119, "FuelTimeGen": 0.001, "ApplianceTimeGen": 0.050, "Emissions": 0.274, "Deforestation": 0.000},
    {"Technologies_id": 5, "Tech_name": "Biogas", "FuelPrice": 0.170, "AppliancePrice": 0.012, "Health": 0.563, "FuelTimeGen": 0.039, "ApplianceTimeGen": 0.050, "Emissions": 0.317, "Deforestation": 0.000},
    {"Technologies_id": 6, "Tech_name": "Charc0", "FuelPrice": 0.151, "AppliancePrice": 0.000, "Health": 0.762, "FuelTimeGen": 0.023, "ApplianceTimeGen": 1.000, "Emissions": 3.374, "Deforestation": 4.009},
    {"Technologies_id": 7, "Tech_name": "Charc1", "FuelPrice": 0.088, "AppliancePrice": 0.014, "Health": 0.716, "FuelTimeGen": 0.013, "ApplianceTimeGen": 0.833, "Emissions": 2.128, "Deforestation": 2.339},
    {"Technologies_id": 8, "Tech_name": "Charc2", "FuelPrice": 0.063, "AppliancePrice": 0.014, "Health": 0.443, "FuelTimeGen": 0.010, "ApplianceTimeGen": 0.667, "Emissions": 1.532, "Deforestation": 1.684},
    {"Technologies_id": 9, "Tech_name": "CFirew0", "FuelPrice": 0.166, "AppliancePrice": 0.000, "Health": 1.000, "FuelTimeGen": 0.115, "ApplianceTimeGen": 1.000, "Emissions": 5.670, "Deforestation": 2.024},
    {"Technologies_id": 10, "Tech_name": "CFirew1", "FuelPrice": 0.083, "AppliancePrice": 0.011, "Health": 0.925, "FuelTimeGen": 0.058, "ApplianceTimeGen": 0.833, "Emissions": 2.814, "Deforestation": 1.012},
    {"Technologies_id": 11, "Tech_name": "CFirew2", "FuelPrice": 0.055, "AppliancePrice": 0.011, "Health": 0.849, "FuelTimeGen": 0.038, "ApplianceTimeGen": 0.667, "Emissions": 1.826, "Deforestation": 0.675},
    {"Technologies_id": 12, "Tech_name": "FrBio0", "FuelPrice": 0.000, "AppliancePrice": 0.000, "Health": 1.000, "FuelTimeGen": 2.696, "ApplianceTimeGen": 1.000, "Emissions": 4.588, "Deforestation": 0.000},
    {"Technologies_id": 13, "Tech_name": "FrBio1", "FuelPrice": 0.000, "AppliancePrice": 0.011, "Health": 0.925, "FuelTimeGen": 1.348, "ApplianceTimeGen": 0.833, "Emissions": 2.272, "Deforestation": 0.000},
    {"Technologies_id": 14, "Tech_name": "FrBio2", "FuelPrice": 0.000, "AppliancePrice": 0.011, "Health": 0.849, "FuelTimeGen": 0.899, "ApplianceTimeGen": 0.667, "Emissions": 1.465, "Deforestation": 0.000},
    {"Technologies_id": 15, "Tech_name": "Pellets", "FuelPrice": 0.066, "AppliancePrice": 0.013, "Health": 0.661, "FuelTimeGen": 0.010, "ApplianceTimeGen": 0.333, "Emissions": 0.397, "Deforestation": 0.253},
    {"Technologies_id": 16, "Tech_name": "FFirew0", "FuelPrice": 0.000, "AppliancePrice": 0.000, "Health": 1.000, "FuelTimeGen": 1.923, "ApplianceTimeGen": 1.000, "Emissions": 4.632, "Deforestation": 1.962},
    {"Technologies_id": 17, "Tech_name": "FFirew1", "FuelPrice": 0.000, "AppliancePrice": 0.011, "Health": 0.925, "FuelTimeGen": 0.962, "ApplianceTimeGen": 0.833, "Emissions": 2.295, "Deforestation": 0.981},
    {"Technologies_id": 18, "Tech_name": "FFirew2", "FuelPrice": 0.000, "AppliancePrice": 0.011, "Health": 0.849, "FuelTimeGen": 0.641, "ApplianceTimeGen": 0.667, "Emissions": 1.480, "Deforestation": 0.654},
    {"Technologies_id": 19, "Tech_name": "Ethanol", "FuelPrice": 0.111, "AppliancePrice": 0.007, "Health": 0.390, "FuelTimeGen": 0.004, "ApplianceTimeGen": 0.050, "Emissions": 0.465, "Deforestation": 0.000}
])
# Añadir columnas necesarias al DataFrame
technologies_df["Fuel_id"] = [
    1, 1, 1, 2, 7, 3, 3, 3,
    4, 4, 4, 8, 8, 8, 5, 6, 6, 6, 9
]

technologies_df["Appliance_id"] = [
    1, 2, 3, 4, 5, 6, 7, 8,
    9, 10, 11, 9, 10, 11, 12, 9, 10, 11, 13
]


# --- Estado y crecimiento ---
state = {
    'year': 2020,
    'semester': 'first',
    'price_plan': {
        'details': {
            'fuel_price': [{
                'Dis-Electric': 1.00,
                'LPG': 1.05,
                'Charcoal': 1.10,
                'Biogas': 0.80,
                'Com. Firew.': 1.10,
                'Free Biom.': 1.00,
                'Pellets': 0.80,
                'Free. Firew.': 1.00,
                'Ethanol': 0.80
            }],
            'app_price': [{
                'El0': 1.00,
                'El1': 0.50,
                'El2': 0.50,
                'LPG': 0.50,
                'Biogas': 0.50,
                'Charc0': 1.00,
                'Charc1': 1.00,
                'Charc2': 0.50,
                'CFirew0': 1.00,
                'CFirew1': 1.00,
                'CFirew2': 0.50,
                'Pellets': 0.50,
                'Ethanol': 0.50
            }]
        }
    },
    'deployment_plan': {
        'details': {
            'appliances_max_cap': [{
                'Is_Urban': False,
                1: 0.03,  # El0
                2: 0.02,  # El1
                3: 0.01,  # El2
                4: 0.10,  # LPG
                5: 0.05,  # Biogas
                6: 1.00,  # Charc0
                7: 0.05,  # Charc1
                8: 0.02,  # Charc2
                9: 1.00,  # CFirew0
                10: 0.40, # CFirew1
                11: 0.10, # CFirew2
                15: 0.005, # Pellets
                19: 0.05   # Ethanol
            }],
            'fuel_max_cap': [{
                'Is_Urban': False,
                1: 0.50,  # Dis-Electric
                2: 0.10,  # LPG
                3: 0.05,  # Biogas
                4: 0.15,  # Charcoal
                5: 0.65,  # Com. Firew.
                6: 0.10,  # Free Biom.
                7: 0.005, # Pellets
                8: 0.32,  # Free. Firew.
                9: 0.05   # Ethanol
            }]
        }
    }
}


growth = {
    'base_year': 2020,
    'dem_mult': [{'Electricity': 1, 'Cooking': 1, 'Heating': 1}],
    'dem_elast': [{'Electricity': 0.01}],
    'soc_clus': [{
        'Population_growth': 1.0,
        'Income_growth': 1.0,
        'Technology_progress': 1.0,
        'Social_weight': 1.0,
        'Health': 1.0,
        'Time_gen': 1.0,
        'Emissions': 1.0,
        'Deforestation': 1.0
    }]
}

# --- Área de demanda ---
class DemandAreaMock:
    def __init__(self, id, area_type, data):
        self.id = id
        self.area_type = area_type
        self.data = data

# Valores desde tu Excel
initial_adoptions = { 
    'El0':     0.002,
    'El1':     0.0001,
    'El2':     0.00005,
    'LPG':     0.02,
    'Biogas':  0.0002,
    'Charc0':  0.02,
    'Charc1':  0.05,# -- MAL
    'Charc2':  0.004, #-- MAL
    'CFirew0': 0.31, 
    'CFirew1': 0.226,
    'CFirew2': 0.027,
    'FrBio0':  0.03,
    'FrBio1':  0.029,
    'FrBio2':  0.014,
    'Pellets': 0.003,
    'FFirew0': 0.15,
    'FFirew1': 0.1,
    'FFirew2': 0.01465,
    'Ethanol': 0.0
}

aggregated_params = {
    'DemandMult': {},  # Se rellenará en _update_social_params
    'e_elast_demand': 1.0,
    'Population': 1,
    'will_pay': 0.07,
    'invest_cap': 20.0,
    'change_fact': 0.20,
    'better_fact': 4.0,
    'worse_fact': 6.0,
    'social_weight': 0.35,
    'social_balance': {
        'health': 0.20,
        'time_gender': 0.70,
        'emissions': 0.05,
        'deforestation': 0.05
    }
}

# Time gender para área rural basado en la imagen proporcionada
time_gender_series = pd.Series({
    'El0':     0.017,
    'El1':     0.017,
    'El2':     0.017,
    'LPG':      0.051,
    'Biogas':   0.244,
    'Charc0':  1.125,
    'Charc1':  0.931,
    'Charc2':  0.744,
    'CFirew0': 1.004,
    'CFirew1': 0.802,
    'CFirew2': 0.635,
    'FrBio0':  9.089,
    'FrBio1':  4.878,
    'FrBio2':  3.363,
    'Pellets': 0.343,
    'FFirew0': 4.366,
    'FFirew1': 2.516,
    'FFirew2': 1.789,
    'Ethanol': 0.054
})



demand_area = DemandAreaMock(
    id=1,
    area_type="rural",
    data={
        'initial_adoptions': {
            'rural': initial_adoptions
        },
        'aggregated_clusters': {
            'rural': {
                'params': aggregated_params
            }
        },
        'time_gen_modified': {
            'rural': time_gender_series
        }
    }
)







In [9]:
# --- Ejecutamos el modelo paso a paso ---
model = AdoptionModel2_0_Debug(
    state=state,
    growth=growth,
    demand_area=demand_area,
    technologies_df=technologies_df,
    fuel_price_plan=state['price_plan']['details']['fuel_price'],
    app_price_plan=state['price_plan']['details']['app_price'],
    deploy_caps=state['deployment_plan']['details']
)

area = "rural"

# Paso 1: actualizar parámetros sociales
social_params = model._update_social_params(area)
print("Parámetros sociales actualizados:")
print(social_params)




Parámetros sociales actualizados:
{'DemandMult': {'Electricity': 1.0, 'Cooking': 1.0, 'Heating': 1.0}, 'e_elast_demand': 1.0, 'Population': 1.0, 'will_pay': 0.07, 'invest_cap': 20.0, 'change_fact': 0.2, 'better_fact': 4.0, 'worse_fact': 6.0, 'social_weight': 0.35, 'social_balance': {'health': 0.2, 'time_gender': 0.7, 'emissions': 0.05, 'deforestation': 0.050000000000000044}}


In [10]:
# model.fuel_id_to_name = {
#     1: "Dis-Electric",
#     2: "LPG",
#     3: "Biogas",
#     4: "Charcoal",
#     5: "Com. Firew.",
#     6: "Free Biom.",
#     7: "Pellets",
#     8: "Free. Firew.",
#     9: "Ethanol"
# }
model.fuel_id_to_name = {
    1: "Dis-Electric",
    2: "LPG",
    3: "Charcoal",
    4: "Com. Firew.",
    5: "Pellets",
    6: "Free-Firewood",
    7: "Biogas",
    8: "Free-Biomass",
    9: "Ethanol"
}


model.appl_id_to_name = {
    1: "El0",
    2: "El1",
    3: "El2",
    4: "LPG",
    5: "Biogas",
    6: "Charc0",
    7: "Charc1",
    8: "Charc2",
    9: "CFirew0",
    10: "CFirew1",
    11: "CFirew2",
    12: "Pellets",
    13: "Ethanol"
}


model.demand_area.data["biomass_multipliers"] = {
    "loc_price": {
        "Charcoal": {
            "rural": 1.00
        },
        "Com. Firew.": {
            "rural": 0.80
        },
        # "Free-Firewood": {
        #     "rural": 1.75
        # },
        # "Biogas": {
        #     "rural": 5.00
        # },
        # # Puedes aplicar "Other" a lo que no esté cubierto, por ejemplo:
        # "Pellets": {
        #     "rural": 3.00
        # },
        # "Free-Biomass": {
        #     "rural": 3.00
        # },
        # "Ethanol": {
        #     "rural": 3.00
        # }
    }
}





In [11]:
# Paso 2: indicadores
indicators_df = model._compute_indicators(area, social_params)
print("\nIndicadores calculados (C, S):")
print(indicators_df)



INFO:root:Adopciones iniciales: [2.000e-03 1.000e-04 5.000e-05 2.000e-02 2.000e-04 2.000e-02 5.000e-02
 4.000e-03 3.100e-01 2.260e-01 2.700e-02 3.000e-02 2.900e-02 1.400e-02
 3.000e-03 1.500e-01 1.000e-01 1.465e-02 0.000e+00]


Ref: %s [0.92290945 1.46678755 3.80019415 1.4809831 ]

Indicadores calculados (C, S):
                        C         S          W
Technologies_id                               
1                 0.22680  0.008938  34.393527
2                 0.16980  0.008109  46.957718
3                 0.13480  0.007556  59.787729
4                 0.15510  0.047444  42.666914
5                 0.48360  0.157677  10.778355
6                 1.74110  0.706955   0.877710
7                 1.41420  0.561564   0.838652
8                 1.11790  0.412631   2.225420
9                 1.55168  0.754287   0.186785
10                1.20684  0.587007   1.382220
11                0.94290  0.480327   2.799885
12               12.72460  1.585244   0.030914
13                6.84020  1.002296   0.064970
14                4.71370  0.757533   0.108783
15                0.53950  0.278410   7.840269
16                6.11240  1.378465   0.070245
17                3.53340  0.899096   0.157867
18                2.5

In [12]:
# Paso 3: candidatos iniciales
init_candidates = model._compute_init_candidates(indicators_df, area)
print("\nCandidatos iniciales de adopción:")
print(init_candidates)




Candidatos iniciales de adopción:
Technologies_id
1     0.017051
2     0.021169
3     0.026891
4     0.035203
5     0.005001
6     0.016510
7     0.045473
8     0.004221
9     0.281822
10    0.205485
11    0.023013
12    0.024196
13    0.023405
14    0.011329
15    0.005935
16    0.136365
17    0.090913
18    0.011946
19    0.014070
Name: W, dtype: float64


In [13]:
# Paso 4: adopción LP
final_adoption = model._adoption_lp(init_candidates, area)
print("\nAdopción final (LP):")
print(final_adoption)


Adopción final (LP):
1     0.017051
2     0.021169
3     0.026891
4     0.035203
5     0.005001
6     0.016510
7     0.045473
8     0.004221
9     0.281822
10    0.205485
11    0.023013
12    0.024196
13    0.023405
14    0.011329
15    0.005935
16    0.136365
17    0.090913
18    0.011946
19    0.014070
dtype: float64


In [23]:

# Entradas de ejemplo para probar
state = {'year': 2025}
growth = {
    'base_year': 2020,
    'dem_mult': [{ 'Electricity': 0.02, 'Cooking': 0.01, 'Heating': 0.01 }, { 'Electricity': 0.03, 'Cooking': 0.02, 'Heating': 0.01 }],
    'dem_elast': [{ 'Electricity': 0.01 }, { 'Electricity': 0.02 }],
    'soc_clus': [{
        'Population_growth': 0.01, 'Income_growth': 0.01, 'Technology_progress': 0.01,
        'Social_weight': 0.01, 'Health': 0.01, 'Time_gen': 0.01, 'Emissions': 0.01
    }, {
        'Population_growth': 0.015, 'Income_growth': 0.015, 'Technology_progress': 0.015,
        'Social_weight': 0.015, 'Health': 0.015, 'Time_gen': 0.015, 'Emissions': 0.015
    }]
}

technologies_df = pd.DataFrame([
    {'Technologies_id': 1, 'Tech_name': 'TechA', 'Health': 0.8, 'FuelTimeGen': 0.5,
     'ApplianceTimeGen': 0.5, 'Emissions': 0.3, 'Deforestation': 0.2,
     'Fuel_id': 1, 'Appliance_id': 1, 'FuelPrice': 100, 'AppliancePrice': 50},
    {'Technologies_id': 2, 'Tech_name': 'TechB', 'Health': 0.9, 'FuelTimeGen': 0.4,
     'ApplianceTimeGen': 0.4, 'Emissions': 0.2, 'Deforestation': 0.1,
     'Fuel_id': 2, 'Appliance_id': 2, 'FuelPrice': 80, 'AppliancePrice': 60}
     
])

demand_area = {
    'initial_adoptions': {'rural': {'TechA': 0.1, 'TechB': 0.2}},
    'aggregated_clusters': {
        'rural': {'params': {
            'DemandMult': {}, 'e_elast_demand': 1.0,
            'Population': 100, 'will_pay': 1.0, 'invest_cap': 1.0,
            'change_fact': 0.8, 'better_fact': 1.0, 'worse_fact': 1.0,
            'social_weight': 1.0,
            'social_balance': {'health': 0.25, 'time_gender': 0.25, 'emissions': 0.25, 'deforestation': 0.25}
        }}
    }
}


In [4]:

# Instancia y prueba del modelo
model = AdoptionModel2_0_Debug(
    state=state,
    growth=growth,
    demand_area=demand_area,
    technologies_df=technologies_df
)

# Paso 1: actualizar parámetros sociales
params = model._update_social_params('rural')
params


{'DemandMult': {'Electricity': 1.1040808032,
  'Cooking': 1.0510100501000001,
  'Heating': 1.0510100501000001},
 'e_elast_demand': 1.0510100501000001,
 'Population': 105.10100501000001,
 'will_pay': 1.0510100501000001,
 'invest_cap': 1.0510100501000001,
 'change_fact': 0.8408080400800002,
 'better_fact': 1.0510100501000001,
 'worse_fact': 1.0510100501000001,
 'social_weight': 1.0,
 'social_balance': {'health': 0.26275251252500004,
  'time_gender': 0.26275251252500004,
  'emissions': 0.26275251252500004,
  'deforestation': 0.21174246242499994}}

In [5]:

# Paso 2: calcular indicadores
indicators = model._compute_indicators('rural', params)
indicators


,C,S
Technologies_id,,
1,151.104622,4.003850
2,141.104622,2.998075
